# Long-term Power Quality Trends

This notebook loads multiple PMon parquet files across a time span and analyzes
trends in voltage, frequency, power, and power factor.

**Sections:**
1. Select and load multiple PMon files
2. Voltage trends with statistical bands
3. Frequency stability
4. Power consumption
5. Reactive power and power factor
6. Statistical summary

## 1. Setup and load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib.dates import DateFormatter
%matplotlib inline

In [ ]:
# Data directory
pmon_dir = Path('/var/lib/eq-synapse/data/pmon')  # Adjust as needed

# List all available files
all_files = sorted(pmon_dir.glob('*.parquet'))
print(f"Found {len(all_files)} PMon files")
if all_files:
    print(f"  First: {all_files[0].name}")
    print(f"  Last:  {all_files[-1].name}")

In [ ]:
# Select a range of files to load.
# Option 1: Load the N most recent files
n_files = min(24, len(all_files))  # e.g., last 24 hours
selected_files = all_files[-n_files:]

# Option 2: Filter by date prefix (uncomment and adjust)
# selected_files = [f for f in all_files if f.name.startswith('202501')]

print(f"Loading {len(selected_files)} files...")

In [ ]:
# Load and concatenate all selected files
tables = []
for f in selected_files:
    try:
        tables.append(pq.read_table(f))
    except Exception as e:
        print(f"  Skipping {f.name}: {e}")

if not tables:
    raise RuntimeError("No data loaded. Check the file path and selection.")

data = pa.concat_tables(tables)
time_col = np.array(data['time_us'], dtype='datetime64[us]')

print(f"Loaded {len(data):,} rows from {len(tables)} files")
print(f"Time range: {time_col[0]} to {time_col[-1]}")

## 2. Voltage trends

RMS voltage over time with mean and +/- 1 standard deviation bands.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_title('RMS Voltage Trends')

for col, color, label in [('AVRMS', 'black', 'A'), ('BVRMS', 'red', 'B'), ('CVRMS', 'blue', 'C')]:
    values = np.array(data[col])
    ax.plot(time_col, values, color=color, alpha=0.5, linewidth=0.3, label=label)

    # Add mean line and +/- 1 std deviation band
    mean_val = np.nanmean(values)
    std_val = np.nanstd(values)
    ax.axhline(mean_val, color=color, linestyle='--', alpha=0.4, linewidth=0.8)
    ax.axhspan(mean_val - std_val, mean_val + std_val, color=color, alpha=0.05)

ax.set_ylabel('RMS Voltage (V)')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M'))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Frequency stability

System frequency over time, with deviation from the nominal 60 Hz.

In [ ]:
freq = np.array(data['FREQ'])
nominal = 60.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.set_title('System Frequency')
ax1.plot(time_col, freq, 'k-', linewidth=0.3, alpha=0.7)
ax1.axhline(nominal, color='green', linestyle='--', alpha=0.5, label=f'Nominal ({nominal} Hz)')
ax1.set_ylabel('Frequency (Hz)')
ax1.legend()
ax1.grid(True, alpha=0.2)

ax2.set_title('Frequency Deviation from Nominal')
deviation = freq - nominal
ax2.plot(time_col, deviation * 1000, 'k-', linewidth=0.3, alpha=0.7)  # Convert to mHz
ax2.axhline(0, color='green', linestyle='--', alpha=0.5)
ax2.set_ylabel('Deviation (mHz)')
ax2.set_xlabel('Time (UTC)')
ax2.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M'))
plt.xticks(rotation=45)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Frequency: mean={np.nanmean(freq):.4f} Hz, "
      f"std={np.nanstd(freq)*1000:.2f} mHz, "
      f"min={np.nanmin(freq):.4f} Hz, max={np.nanmax(freq):.4f} Hz")

## 4. Power consumption

Active power per phase and total, with energy calculation over the loaded time span.

In [ ]:
awatt = np.array(data['AWATT'])
bwatt = np.array(data['BWATT'])
cwatt = np.array(data['CWATT'])
total_watts = awatt + bwatt + cwatt

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_title('Active Power')
ax.plot(time_col, awatt, 'k', alpha=0.5, linewidth=0.3, label='A')
ax.plot(time_col, bwatt, 'r', alpha=0.5, linewidth=0.3, label='B')
ax.plot(time_col, cwatt, 'b', alpha=0.5, linewidth=0.3, label='C')
ax.plot(time_col, total_watts, 'green', alpha=0.7, linewidth=0.5, label='Total')
ax.set_ylabel('Power (W)')
ax.set_xlabel('Time (UTC)')
ax.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M'))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Energy calculation
# PMon records are approximately once per line cycle (~1/60 s)
# Total energy = sum(power * dt)
time_seconds = (time_col[-1] - time_col[0]) / np.timedelta64(1, 's')
dt = time_seconds / len(total_watts)  # Average sample interval

energy_wh = np.nansum(total_watts) * dt / 3600  # Convert W*s to Wh
energy_kwh = energy_wh / 1000

print(f"Time span: {time_seconds/3600:.1f} hours")
print(f"Average power: {np.nanmean(total_watts):.0f} W")
print(f"Peak power: {np.nanmax(total_watts):.0f} W")
print(f"Total energy: {energy_kwh:.2f} kWh")

## 5. Reactive power and power factor

Compute power factor from the fundamental active power (AFWATT) and reactive
power (AFVAR) columns:

$$PF = \frac{P}{\sqrt{P^2 + Q^2}}$$

In [ ]:
# Fundamental active and reactive power
afwatt = np.array(data['AFWATT'])
bfwatt = np.array(data['BFWATT'])
cfwatt = np.array(data['CFWATT'])
afvar = np.array(data['AFVAR'])
bfvar = np.array(data['BFVAR'])
cfvar = np.array(data['CFVAR'])

total_p = afwatt + bfwatt + cfwatt
total_q = afvar + bfvar + cfvar
apparent = np.sqrt(total_p**2 + total_q**2)

# Power factor (avoid division by zero)
pf = np.where(apparent > 0, np.abs(total_p) / apparent, 0.0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.set_title('Reactive Power (Fundamental)')
ax1.plot(time_col, total_q, 'purple', alpha=0.5, linewidth=0.3)
ax1.set_ylabel('Reactive Power (VAR)')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.2)

ax2.set_title('Power Factor')
ax2.plot(time_col, pf, 'green', alpha=0.5, linewidth=0.3)
ax2.set_ylabel('Power Factor')
ax2.set_ylim(0, 1.05)
ax2.axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='0.9 reference')
ax2.set_xlabel('Time (UTC)')
ax2.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M'))
plt.xticks(rotation=45)
ax2.legend()
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 6. Statistical summary

A comprehensive table of min, max, mean, and standard deviation for all key metrics.

In [ ]:
metrics = {
    'FREQ (Hz)': freq,
    'AVRMS (V)': np.array(data['AVRMS']),
    'BVRMS (V)': np.array(data['BVRMS']),
    'CVRMS (V)': np.array(data['CVRMS']),
    'AIRMS (A)': np.array(data['AIRMS']),
    'BIRMS (A)': np.array(data['BIRMS']),
    'CIRMS (A)': np.array(data['CIRMS']),
    'Total P (W)': total_watts,
    'Total Q (VAR)': total_q,
    'PF': pf,
}

print(f"{'Metric':<16} {'Min':>12} {'Max':>12} {'Mean':>12} {'Std':>12}")
print('-' * 66)
for name, values in metrics.items():
    print(f"{name:<16} {np.nanmin(values):>12.4f} {np.nanmax(values):>12.4f} "
          f"{np.nanmean(values):>12.4f} {np.nanstd(values):>12.4f}")

In [ ]:
# Export summary to CSV (optional)
# import csv
# with open('power_quality_summary.csv', 'w', newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(['Metric', 'Min', 'Max', 'Mean', 'Std'])
#     for name, values in metrics.items():
#         writer.writerow([name, np.nanmin(values), np.nanmax(values),
#                          np.nanmean(values), np.nanstd(values)])
# print('Saved to power_quality_summary.csv')